# RSA semantic sidecar: BYO index + stage-level systems benchmark

This notebook builds the portable **Binary1-LS2-int4** sidecar from real held-out fashion embeddings/programs and measures the semantic search stage with the native Rust executor.

Measured here: **semantic scoring + calibrated composition + top-k + optional safe early exit**.

Not measured here: ANN traversal, exact filters, RPC/networking, or the downstream ranker.


In [ ]:
#@title 1) Settings
FULL_DATA = False #@param {type:"boolean"}
RESIDENT_ITEMS = 500000 #@param {type:"integer"}
REPEATS = 11 #@param {type:"integer"}
MIXED_SIGNS = True #@param {type:"boolean"}

print('FULL_DATA:', FULL_DATA)
print('RESIDENT_ITEMS:', RESIDENT_ITEMS)


In [ ]:
#@title 2) Clone repo and install dependencies
import os, pathlib, shutil, subprocess
ROOT = pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['pip','install','-q','-e','.','faiss-cpu'], check=True)
print('commit:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
print('rustc:', subprocess.check_output(['rustc','--version']).decode().strip())


In [ ]:
#@title 3) Build portable held-out sidecar index and programs
import time, os, subprocess
os.chdir('/content/ras')
cfg = 'configs/binary_bbq.yaml' if FULL_DATA else 'configs/binary_bbq_smoke.yaml'
OUT = '/content/native_sidecar'
t0=time.time()
subprocess.run(['python','-m','experiments.export_native_finalists','--config',cfg,'--out-dir',OUT], check=True)
print(f'export finished in {(time.time()-t0)/60:.1f} min')
print('sidecar index:', OUT + '/sidecar_index')
print('program store:', OUT + '/sidecar_programs')


In [ ]:
#@title 4) Sanity-check Python BYO inference
import numpy as np
from ras import SemanticExecutor
executor = SemanticExecutor.open(OUT + '/sidecar_index', OUT + '/sidecar_programs')
print('programs:', executor.programs.names())
n=min(5000, executor.index.n_items)
ids=np.arange(n, dtype=np.int64)
names=executor.programs.names()
pos=names[:2]
neg=names[2:3]
r=executor.topk(ids, positive=pos, negative=neg, k=max(1,n//5))
print('query:', '+', pos, '-', neg)
print('semantic_ms:', round(r.semantic_ms,3), 'topk_ms:', round(r.topk_ms,3))
print('top rows:', r.row_ids[:10])


In [ ]:
#@title 5) Run native stage-level latency benchmark
import subprocess, os, time
cmd=[
 'python','-m','experiments.sidecar_systems',
 '--index',OUT+'/sidecar_index',
 '--programs',OUT+'/sidecar_programs',
 '--out-dir','/content/sidecar_systems',
 '--resident-items',str(RESIDENT_ITEMS),
 '--candidate-counts','5000,20000,100000',
 '--predicate-counts','1,2,4,8',
 '--keep-fraction','0.2',
 '--repeats',str(REPEATS),
]
if MIXED_SIGNS: cmd.append('--mixed-signs')
t0=time.time()
subprocess.run(cmd, check=True)
print(f'benchmark finished in {time.time()-t0:.1f}s')


In [ ]:
#@title 6) Inspect latency and early-exit savings
import pandas as pd, json
df=pd.read_csv('/content/sidecar_systems/sidecar_latency.csv')
cols=['requested_candidates','requested_predicates','early_exit_requested','median_ms','p95_ms','million_candidates_per_s','evaluated_predicate_fraction','early_reject_fraction']
display(df[cols].sort_values(['requested_candidates','requested_predicates','early_exit_requested']))

base=df[df.early_exit_requested==False][['requested_candidates','requested_predicates','median_ms']].rename(columns={'median_ms':'median_no_early'})
early=df[df.early_exit_requested==True][['requested_candidates','requested_predicates','median_ms','p95_ms','evaluated_predicate_fraction','early_reject_fraction']]
cmp=early.merge(base,on=['requested_candidates','requested_predicates'])
cmp['speedup']=cmp.median_no_early/cmp.median_ms
display(cmp.sort_values(['requested_candidates','requested_predicates']))
print(json.load(open('/content/sidecar_systems/environment.json')))


In [ ]:
#@title 7) Package results
import shutil, os
shutil.make_archive('/content/rsa_sidecar_systems','zip','/content/sidecar_systems')
print('/content/rsa_sidecar_systems.zip')
